# Example Usage: Direct Getter Methods for Patient Data

## Overview

This notebook demonstrates how to use the dot notation getter methods in `pat2vec_obj` to easily access patient data without guessing table names or column structures.

The pat2vec library provides:
- **Direct Epic Access** (requires Elasticsearch with `cogstack=True`)
- **Database Backend** (raw data from batches)
- **Feature Vectors** (processed feature vectors)

In [ ]:
# First, let's set up a pat2vec object with database backend

import os
import random
import shutil
from datetime import datetime

from dateutil.relativedelta import relativedelta

# Fix the random seed for reproducibility
random_seed_value = 42
random.seed(random_seed_value)

# Clean up previous outputs
clear_previous_outputs = True
if clear_previous_outputs:
    shutil.rmtree("new_project", ignore_errors=True)

# Set up a temporary database path
DB_FILENAME = "temp_example_db.sqlite"
DB_PATH = os.path.join("new_project", "outputs", DB_FILENAME)
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    os.remove(DB_PATH)
except FileNotFoundError:
    pass

db_connection_string = f"sqlite:///{DB_PATH}"

print(f"Database path: {DB_PATH}")

In [ ]:
# Import pat2vec and create config

from pat2vec.main_pat2vec import main
from pat2vec.util.config_pat2vec import config_class

# Configuration settings
main_options_dict = {
    "demo": True,  # Enable demographic information (Ethnicity mapped to UK census categories, age, death).
    "bmi": True,  # Enable BMI (Body Mass Index) information.
    "bloods": True,  # Enable blood-related information
    "drugs": True,  # Enable drug-related information
    "diagnostics": True,  # Enable diagnostic information
    "core_02": True,  # Enable core_02 information
    "bed": True,  # Enable bed n information
    "vte_status": True,  # Enable VTE () status information
    "hosp_site": True,  # Enable hospital site information
    "core_resus": True,  # Enable core resuscitation information
    "news": True,  # Enable NEWS (National Early Warning Score) information
    "smoking": True,  # Enable smoking-related information
    "annotations": True,  # Enable EPR documents annotations via MedCat
    "annotations_mrc": True,  # Enable MRC (Additional clinical note observations index) annotations via MedCat
    "negated_presence_annotations": False,  # Enable or disable negated presence annotations
    "appointments": False,  # Enable appointments information
    "annotations_reports": True,  # Enable reports information
    "textual_obs": True,  # Enable textual observations (basic_observations index) annotations via MedCat
    "covid": True,  # Enable covid test results
    "epic_encounters": True,  # Enable epic encounters data (patient encounters/admissions)
    "epic_clinical_notes": True,  # Enable epic clinical notes data
    "epic_medical_history": True,  # Enable epic medical history data
    "epic_orders": True,  # Enable epic orders data (lab orders, medication orders, etc.)
    "epic_lab_results": True,  # Enable epic lab results data
    "epic_patients": True,  # Enable epic patient demographic data
    "epic_imaging_reports": True,  # Enable epic imaging reports data
    "epic_clinical_notes_appointments": True,  # Enable epic clinical notes appointments data
}

config_obj = config_class(
    proj_name="new_project",
    current_path_dir="",
    main_options=main_options_dict,
    start_date=(datetime(1995, 1, 1)),
    years=30,
    months=0,
    days=0,
    batch_mode=True,
    store_annot=False,
    multi_process=False,
    strip_list=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,  # Use dummy data for testing
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=datetime.now(),
    patient_id_column_name="auto",
    global_start_year=1995,
    global_start_month=1,
    global_end_year=2025,
    global_end_month=1,
    global_start_day=1,
    global_end_day=1,
    time_window_interval_delta=relativedelta(years=31),
    split_clinical_notes=False,
    lookback=False,
    add_icd10=False,
    add_opc4s=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
)

# Create pat2vec object with testing mode
pat2vec_obj = main(
    cogstack=True,  # Set to True for Elasticsearch access
    use_filter=False,
    json_filter_path=None,
    random_seed_val=random_seed_value,
    hostname=None,
    config_obj=config_obj,
)

In [ ]:
# Process all patients to populate database tables
# This is required for database backend - data is only saved when pat_maker(i) is called
from tqdm import tqdm

print(f"Processing {len(pat2vec_obj.all_patient_list)} patients...")
for i in tqdm(range(len(pat2vec_obj.all_patient_list))):
    try:
        pat2vec_obj.pat_maker(i)
    except Exception as e:
        print(f"Error processing patient at index {i}: {e}")

print("Patient processing complete!")

## Direct Epic Access (requires Elasticsearch with cogstack=True)

These methods query the Epic database directly via Elasticsearch. Note: They require `cogstack=True` and valid Elasticsearch credentials.

In [ ]:
# Get epic patients data (requires Elasticsearch with cogstack=True)

# Note: This requires a real Elasticsearch connection to Epic
# For demonstration, we'll skip if testing mode is active

if not config_obj.testing:
    try:
        # Get data for the first patient (requires patient durable key)
        epic_patients = pat2vec_obj.get_epic_patients(
            patient_id=pat2vec_obj.all_patient_list[0], start_year=1995, end_year=2025
        )
        print(f"Epic patients data shape: {epic_patients.shape}")
        print(epic_patients.head())
    except Exception as e:
        print(f"get_epic_patents failed (expected in testing mode): {e}")

else:
    print("Epic data access requires cogstack=True with real Elasticsearch credentials")

In [ ]:
# Get epic encounters (requires Elasticsearch)

if not config_obj.testing:
    try:
        epic_encounters = pat2vec_obj.get_epic_encounters(
            patient_id=pat2vec_obj.all_patient_list[0], start_year=1995, end_year=2025
        )
        print(f"Epic encounters data shape: {epic_encounters.shape}")
    except Exception as e:
        print(f"get_epic_encounters failed (expected in testing mode): {e}")

In [ ]:
# Get epic lab results (requires Elasticsearch)

if not config_obj.testing:
    try:
        epic_lab_results = pat2vec_obj.get_epic_lab_results(
            patient_id=pat2vec_obj.all_patient_list[0], start_year=1995, end_year=2025
        )
        print(f"Epic lab results data shape: {epic_lab_results.shape}")
    except Exception as e:
        print(f"get_epic_lab_results failed (expected in testing mode): {e}")

## Database Backend Methods (Raw Data)

These methods access raw data stored in the database backend. They work regardless of `cogstack` setting.

In [ ]:
# Get all available patients

print(f"Total patients in cohort: {len(pat2vec_obj.all_patient_list)}")
if len(pat2vec_obj.all_patient_list) > 0:
    sample_patient_id = pat2vec_obj.all_patient_list[0]
    print(f"Sample patient ID: {sample_patient_id}")

In [ ]:
# Access raw drugs data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    try:
        drugs_df = pat2vec_obj.get_raw_drugs(patient_id)
        print(f"Raw drugs data shape: {drugs_df.shape}")
        if not drugs_df.empty:
            print("Sample drugs data:")
            print(drugs_df.head())
        else:
            print("No drug records found for this patient")
    except Exception as e:
        print(f"Error getting raw drugs: {e}")

In [ ]:
# Access raw blood test data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    try:
        bloods_df = pat2vec_obj.get_raw_bloods(patient_id)
        print(f"Raw blood tests data shape: {bloods_df.shape}")
        if not bloods_df.empty:
            print("Sample blood test columns:", bloods_df.columns.tolist())
        else:
            print("No blood test records found for this patient")
    except Exception as e:
        print(f"Error getting raw bloods: {e}")

In [ ]:
# Access raw EPR clinical documents

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    try:
        epr_df = pat2vec_obj.get_raw_epr_docs(patient_id)
        print(f"Raw EPR documents data shape: {epr_df.shape}")
        if not epr_df.empty:
            print("Sample EPR columns:", epr_df.columns.tolist())
        else:
            print("No EPR document records found for this patient")
    except Exception as e:
        print(f"Error getting raw EPR docs: {e}")

In [ ]:
# Access raw demographic data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    try:
        demo_df = pat2vec_obj.get_raw_demographics(patient_id)
        print(f"Raw demographics data shape: {demo_df.shape}")
        if not demo_df.empty:
            print("Sample demographic columns:", demo_df.columns.tolist())
        else:
            print("No demographic records found for this patient")
    except Exception as e:
        print(f"Error getting raw demographics: {e}")

In [ ]:
# Access other raw data tables

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # MCT annotated documents
    try:
        mct_df = pat2vec_obj.get_raw_mct_docs(patient_id)
        print(f"Raw MCT docs shape: {mct_df.shape}")
    except Exception as e:
        print(f"Error getting raw MCT docs: {e}")

    # Textual observations
    try:
        textual_obs_df = pat2vec_obj.get_raw_textual_obs(patient_id)
        print(f"Raw textual obs shape: {textual_obs_df.shape}")
    except Exception as e:
        print(f"Error getting raw textual observations: {e}")

In [ ]:
# Access remaining raw data tables

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # Reports
    try:
        reports_df = pat2vec_obj.get_raw_reports(patient_id)
        print(f"Raw reports shape: {reports_df.shape}")
    except Exception as e:
        print(f"Error getting raw reports: {e}")

    # Diagnostics
    try:
        diagnostics_df = pat2vec_obj.get_raw_diagnostics(patient_id)
        print(f"Raw diagnostics shape: {diagnostics_df.shape}")
    except Exception as e:
        print(f"Error getting raw diagnostics: {e}")

In [ ]:
# Access vital signs and observations data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # NEWS scores
    try:
        news_df = pat2vec_obj.get_raw_news(patient_id)
        print(f"Raw NEWS scores shape: {news_df.shape}")
    except Exception as e:
        print(f"Error getting raw NEWS: {e}")

    # BMI records
    try:
        bmi_df = pat2vec_obj.get_raw_bmi(patient_id)
        print(f"Raw BMI data shape: {bmi_df.shape}")
    except Exception as e:
        print(f"Error getting raw BMI: {e}")

In [ ]:
# Access more clinical data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # Appointments
    try:
        appointments_df = pat2vec_obj.get_raw_appointments(patient_id)
        print(f"Raw appointments shape: {appointments_df.shape}")
    except Exception as e:
        print(f"Error getting raw appointments: {e}")

    # COVID results
    try:
        covid_df = pat2vec_obj.get_raw_covid(patient_id)
        print(f"Raw COVID data shape: {covid_df.shape}")
    except Exception as e:
        print(f"Error getting raw COVID: {e}")

In [ ]:
# Access remaining observation data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # Smoking status
    try:
        smoking_df = pat2vec_obj.get_raw_smoking(patient_id)
        print(f"Raw smoking data shape: {smoking_df.shape}")
    except Exception as e:
        print(f"Error getting raw smoking: {e}")

    # Core 02 (oxygen saturation)
    try:
        core_02_df = pat2vec_obj.get_raw_core_02(patient_id)
        print(f"Raw CORE 02 data shape: {core_02_df.shape}")
    except Exception as e:
        print(f"Error getting raw CORE 02: {e}")

In [ ]:
# Access additional clinical data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # Bed numbers
    try:
        bed_df = pat2vec_obj.get_raw_bed(patient_id)
        print(f"Raw bed data shape: {bed_df.shape}")
    except Exception as e:
        print(f"Error getting raw bed: {e}")

    # VTE status
    try:
        vte_df = pat2vec_obj.get_raw_vte(patient_id)
        print(f"Raw VTE data shape: {vte_df.shape}")
    except Exception as e:
        print(f"Error getting raw VTE: {e}")

In [ ]:
# Access more clinical data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # Hospital sites
    try:
        hospsite_df = pat2vec_obj.get_raw_hospsite(patient_id)
        print(f"Raw hospital site data shape: {hospsite_df.shape}")
    except Exception as e:
        print(f"Error getting raw hospital sites: {e}")

    # Resuscitation status
    try:
        resus_df = pat2vec_obj.get_raw_resus(patient_id)
        print(f"Raw resuscitation data shape: {resus_df.shape}")
    except Exception as e:
        print(f"Error getting raw resuscitation: {e}")

In [ ]:
# Access Epic data from database

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # Epic encounters
    try:
        epic_enc_db_df = pat2vec_obj.get_raw_epic_encounters(patient_id)
        print(f"Raw Epic encounters (DB) shape: {epic_enc_db_df.shape}")
    except Exception as e:
        print(f"Error getting raw epic_encounters from DB: {e}")

    # Epic clinical notes
    try:
        epic_notes_df = pat2vec_obj.get_raw_epic_clinical_notes(patient_id)
        print(f"Raw Epic clinical notes (DB) shape: {epic_notes_df.shape}")
    except Exception as e:
        print(f"Error getting raw epic clinical notes from DB: {e}")

In [ ]:
# Access remaining Epic data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # Epic medical history
    try:
        epic_med_hist_df = pat2vec_obj.get_raw_epic_medical_history(patient_id)
        print(f"Raw Epic medical history (DB) shape: {epic_med_hist_df.shape}")
    except Exception as e:
        print(f"Error getting raw epic medical history from DB: {e}")

    # Epic orders
    try:
        epic_orders_df = pat2vec_obj.get_raw_epic_orders(patient_id)
        print(f"Raw Epic orders (DB) shape: {epic_orders_df.shape}")
    except Exception as e:
        print(f"Error getting raw epic orders from DB: {e}")

In [ ]:
# Access more Epic data

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # Epic lab results
    try:
        epic_lab_db_df = pat2vec_obj.get_raw_epic_lab_results(patient_id)
        print(f"Raw Epic lab results (DB) shape: {epic_lab_db_df.shape}")
    except Exception as e:
        print(f"Error getting raw epic lab results from DB: {e}")

    # Epic patients
    try:
        epic_pat_db_df = pat2vec_obj.get_raw_epic_patients(patient_id)
        print(f"Raw Epic patients (DB) shape: {epic_pat_db_df.shape}")
    except Exception as e:
        print(f"Error getting raw epic patients from DB: {e}")

In [ ]:
# Access general observations

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    # General observations
    try:
        obs_df = pat2vec_obj.get_raw_obs(patient_id)
        print(f"Raw general observations shape: {obs_df.shape}")
    except Exception as e:
        print(f"Error getting raw observations: {e}")

## Feature Vectors Access

These methods access processed feature vectors.

In [ ]:
# Get all processed feature vectors

try:
    all_features = pat2vec_obj.get_all_features()
    print(f"All features data shape: {all_features.shape}")
    if not all_features.empty:
        print("Sample feature columns:", all_features.columns.tolist()[:10])
except Exception as e:
    print(f"Error getting all features: {e}")

In [ ]:
# Get features for a specific patient

if len(pat2vec_obj.all_patient_list) > 0:
    patient_id = pat2vec_obj.all_patient_list[0]

    try:
        patient_features = pat2vec_obj.get_features(patient_id)
        print(f"Patient {patient_id} features shape: {patient_features.shape}")
        if not patient_features.empty:
            print("Sample feature columns:", patient_features.columns.tolist()[:10])
    except Exception as e:
        print(f"Error getting patient features: {e}")

In [ ]:
# Build merged Epic encounters data for all patients

if len(pat2vec_obj.all_patient_list) > 0:
    from pat2vec.util.post_processing_build_methods import merge_epic_encounters_csv

    try:
        merged_encounters_path = merge_epic_encounters_csv(
            all_pat_list=pat2vec_obj.all_patient_list,
            config_obj=config_obj,
            overwrite=False,
        )
        print(f"Merged Epic encounters saved to: {merged_encounters_path}")

        # Load and display a sample
        import pandas as pd

        sample_df = pd.read_csv(merged_encounters_path, nrows=5)
        print("\nSample merged encounters data (first 5 rows):")
        print(sample_df)
        print(f"\nTotal encounters: {len(sample_df)}")
    except Exception as e:
        print(f"Error building merged Epic encounters: {e}")

In [ ]:
# Build merged Epic lab results data for all patients

if len(pat2vec_obj.all_patient_list) > 0:
    from pat2vec.util.post_processing_build_methods import merge_epic_lab_results_csv

    try:
        merged_lab_results_path = merge_epic_lab_results_csv(
            all_pat_list=pat2vec_obj.all_patient_list,
            config_obj=config_obj,
            overwrite=False,
        )
        print(f"Merged Epic lab results saved to: {merged_lab_results_path}")

        # Load and display a sample
        import pandas as pd

        sample_df = pd.read_csv(merged_lab_results_path, nrows=5)
        print("\nSample merged lab results data (first 5 rows):")
        print(sample_df)
        print(f"\nTotal lab results: {len(sample_df)}")
    except Exception as e:
        print(f"Error building merged Epic lab results: {e}")

In [ ]:
# Build merged Epic patients data for all patients

if len(pat2vec_obj.all_patient_list) > 0:
    from pat2vec.util.post_processing_build_methods import merge_epic_patients_csv

    try:
        merged_patients_path = merge_epic_patients_csv(
            all_pat_list=pat2vec_obj.all_patient_list,
            config_obj=config_obj,
            overwrite=False,
        )
        print(f"Merged Epic patients saved to: {merged_patients_path}")

        # Load and display a sample
        import pandas as pd

        sample_df = pd.read_csv(merged_patients_path, nrows=5)
        print("\nSample merged patients data (first 5 rows):")
        print(sample_df)
        print(f"\nTotal patients: {len(sample_df)}")
    except Exception as e:
        print(f"Error building merged Epic patients: {e}")

In [ ]:
# Build merged Epic clinical notes documents for all patients

if len(pat2vec_obj.all_patient_list) > 0:
    from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_doc_df

    try:
        merged_clinical_notes_path = build_merged_epr_mct_doc_df(
            all_pat_list=pat2vec_obj.all_patient_list,
            config_obj=config_obj,
            overwrite=False,
        )
        print(f"Merged Epic clinical notes saved to: {merged_clinical_notes_path}")

        # Load and display a sample
        import pandas as pd

        sample_df = pd.read_csv(merged_clinical_notes_path, nrows=5)
        print("\nSample merged clinical notes data (first 5 rows):")
        print(sample_df)
        print(f"\nTotal clinical notes: {len(sample_df)}")
    except Exception as e:
        print(f"Error building merged Epic clinical notes: {e}")

In [ ]:
# Build merged Epic orders documents for all patients

if len(pat2vec_obj.all_patient_list) > 0:
    from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_doc_df

    try:
        merged_orders_path = build_merged_epr_mct_doc_df(
            all_pat_list=pat2vec_obj.all_patient_list,
            config_obj=config_obj,
            overwrite=False,
        )
        print(f"Merged Epic orders saved to: {merged_orders_path}")

        # Load and display a sample
        import pandas as pd

        sample_df = pd.read_csv(merged_orders_path, nrows=5)
        print("\nSample merged orders data (first 5 rows):")
        print(sample_df)
        print(f"\nTotal orders records: {len(sample_df)}")
    except Exception as e:
        print(f"Error building merged Epic orders: {e}")

In [ ]:
# Build merged Epic medical history documents for all patients

if len(pat2vec_obj.all_patient_list) > 0:
    from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_doc_df

    try:
        merged_medical_history_path = build_merged_epr_mct_doc_df(
            all_pat_list=pat2vec_obj.all_patient_list,
            config_obj=config_obj,
            overwrite=False,
        )
        print(f"Merged Epic medical history saved to: {merged_medical_history_path}")

        # Load and display a sample
        import pandas as pd

        sample_df = pd.read_csv(merged_medical_history_path, nrows=5)
        print("\nSample merged medical history data (first 5 rows):")
        print(sample_df)
        print(f"\nTotal medical history records: {len(sample_df)}")
    except Exception as e:
        print(f"Error building merged Epic medical history: {e}")

In [ ]:
# Build merged Epic imaging reports documents for all patients

if len(pat2vec_obj.all_patient_list) > 0:
    from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_doc_df

    try:
        merged_imaging_reports_path = build_merged_epr_mct_doc_df(
            all_pat_list=pat2vec_obj.all_patient_list,
            config_obj=config_obj,
            overwrite=False,
        )
        print(f"Merged Epic imaging reports saved to: {merged_imaging_reports_path}")

        # Load and display a sample
        import pandas as pd

        sample_df = pd.read_csv(merged_imaging_reports_path, nrows=5)
        print("\nSample merged imaging reports data (first 5 rows):")
        print(sample_df)
        print(f"\nTotal imaging reports: {len(sample_df)}")
    except Exception as e:
        print(f"Error building merged Epic imaging reports: {e}")

In [ ]:
# Build merged Epic clinical notes appointments documents for all patients

if len(pat2vec_obj.all_patient_list) > 0:
    from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_doc_df

    try:
        merged_appointments_path = build_merged_epr_mct_doc_df(
            all_pat_list=pat2vec_obj.all_patient_list,
            config_obj=config_obj,
            overwrite=False,
        )
        print(
            f"Merged Epic clinical notes appointments saved to: {merged_appointments_path}"
        )

        # Load and display a sample
        import pandas as pd

        sample_df = pd.read_csv(merged_appointments_path, nrows=5)
        print("\nSample merged appointments data (first 5 rows):")
        print(sample_df)
        print(f"\nTotal appointments records: {len(sample_df)}")
    except Exception as e:
        print(f"Error building merged Epic clinical notes appointments: {e}")

In [ ]:
# Build merged EPR/MCT annotations data for all patients

if len(pat2vec_obj.all_patient_list) > 0:
    from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_annot_df

    try:
        merged_annotations_path = build_merged_epr_mct_annot_df(
            all_pat_list=pat2vec_obj.all_patient_list,
            config_obj=config_obj,
            overwrite=False,
        )
        print(f"Merged EPR/MCT annotations saved to: {merged_annotations_path}")

        # Load and display a sample
        import pandas as pd

        sample_df = pd.read_csv(merged_annotations_path, nrows=5)
        print("\nSample merged annotations data (first 5 rows):")
        print(sample_df)
        print(f"\nTotal annotation records: {len(sample_df)}")
    except Exception as e:
        print(f"Error building merged EPR/MCT annotations: {e}")

## Summary

### Quick Reference for Getter Methods

**Direct Epic Access (requires Elasticsearch):**
- `get_epic_patients(patient_id, start_year=1995, end_year=2025)`
- `get_epic_encounters(patient_id, start_year=1995, end_year=2025)`
- `get_epic_lab_results(patient_id, start_year=1995, end_year=2025)`

**Database Backend (raw data):**
- `get_raw_drugs(patient_id)` - Drugs
- `get_raw_bloods(patient_id)` - Blood tests
- `get_raw_epr_docs(patient_id)` - EPR clinical documents
- `get_raw_demographics(patient_id)` - Demographics
- `get_raw_mct_docs(patient_id)` - MCT annotated docs
- `get_raw_textual_obs(patient_id)` - Textual observations
- `get_raw_reports(patient_id)` - Medical reports
- `get_raw_diagnostics(patient_id)` - Diagnostic orders
- `get_raw_news(patient_id)` - NEWS scores
- `get_raw_bmi(patient_id)` - BMI records
- `get_raw_appointments(patient_id)` - Appointments
- `get_raw_covid(patient_id)` - COVID test results
- `get_raw_smoking(patient_id)` - Smoking status
- `get_raw_core_02(patient_id)` - Oxygen saturation
- `get_raw_bed(patient_id)` - Bed numbers
- `get_raw_vte(patient_id)` - VTE status
- `get_raw_hospsite(patient_id)` - Hospital sites
- `get_raw_resus(patient_id)` - Resuscitation data
- `get_raw_obs(patient_id)` - General observations

**Epic Data from Database:**
- `get_raw_epic_encounters(patient_id)` - Epic encounters
- `get_raw_epic_clinical_notes(patient_id)` - Epic clinical notes
- `get_raw_epic_medical_history(patient_id)` - Medical history
- `get_raw_epic_orders(patient_id)` - Orders
- `get_raw_epic_lab_results(patient_id)` - Lab results
- `get_raw_epic_patients(patient_id)` - Patient records

**Feature Vectors:**
- `get_all_features()` - All processed feature vectors
- `get_features(patient_id)` - Features for a single patient